In [1]:
# Import Libriary
import snowflake.connector
from dotenv import load_dotenv, find_dotenv
import pandas as pd
import os

In [2]:
# Load dotenv
load_dotenv(find_dotenv('../.env'), override=True)

conn = snowflake.connector.connect(
    account=os.getenv('SNOWFLAKE_ACCOUNT'),
    user=os.getenv('SNOWFLAKE_USER'),
    password=os.getenv('SNOWFLAKE_PASSWORD'),
    warehouse=os.getenv('SNOWFLAKE_WAREHOUSE'),
    database=os.getenv('SNOWFLAKE_DATABASE'),
    schema=os.getenv('SNOWFLAKE_SCHEMA'),
    autocommit=True
)

print("Connected")

Connected


In [3]:
# Load Table
def load_table(conn, table_name, df):
    cur = conn.cursor()
    cols = ', '.join(df.columns)
    placeholders = ', '.join(['%s'] * len(df.columns))
    sql = f'INSERT INTO DENTAL_AI_DB.TIER1_ERP.{table_name} ({cols}) VALUES ({placeholders})'
    data = [tuple(row) for row in df.itertuples(index=False)]
    cur.executemany(sql, data)
    print(f"Loaded {len(df)} rows into {table_name}")
    cur.close()

df_dentists   = pd.read_csv('../data/synthetic/dentists.csv')
df_insurers   = pd.read_csv('../data/synthetic/insurers.csv')
df_patients   = pd.read_csv('../data/synthetic/patients.csv')
df_treatments = pd.read_csv('../data/synthetic/treatments.csv')
df_documents  = pd.read_csv('../data/synthetic/documents.csv')

load_table(conn, 'DENTISTS',            df_dentists)
load_table(conn, 'INSURANCE_COMPANIES', df_insurers)
load_table(conn, 'PATIENTS',            df_patients)
load_table(conn, 'TREATMENTS',          df_treatments)
load_table(conn, 'DOCUMENTS',           df_documents)

cur = conn.cursor()
cur.execute("SELECT COUNT(*) FROM DENTAL_AI_DB.TIER1_ERP.DENTISTS")
print("Verify dentist count:", cur.fetchone())
cur.close()

conn.close()
print("Done")

Loaded 50 rows into DENTISTS
Loaded 10 rows into INSURANCE_COMPANIES
Loaded 5000 rows into PATIENTS
Loaded 15000 rows into TREATMENTS
Loaded 23965 rows into DOCUMENTS
Verify dentist count: (50,)
Done
